# 03 — Generate stores

**Business objective:** stores are the geographic anchor for the entire MVP —
the target question ("why did sales drop in Sydney") is fundamentally a
store/city-level question, so this small table matters more than its size
suggests.

**What we're generating:** 8 stores across 5 Australian cities, with two
stores in Sydney (deliberately — that's the city under investigation) and two
in Melbourne (our main comparison city).

In [1]:
import sys
sys.path.insert(0, '..')
from notebooks import nb_config as cfg

import pandas as pd
import numpy as np

rng = np.random.default_rng(cfg.SEED + 2)

## Generation logic

Each store gets an open date at least a year before the order-history window starts, so no store is "too new" to have a full trading history.

In [2]:
open_start = pd.Timestamp("2020-01-01")
open_end = pd.Timestamp("2022-06-01")
open_range_days = (open_end - open_start).days

rows = []
for i, (city, state) in enumerate(cfg.CITIES, start=1):
    rows.append({
        "store_id": i,
        "store_name": f"{city} Store {i}",
        "city": city,
        "state": state,
        "region": "Australia",
        "open_date": open_start + pd.Timedelta(days=int(rng.uniform(0, open_range_days))),
    })

stores = pd.DataFrame(rows)
stores

,store_id,store_name,city,state,region,open_date
0,1,Sydney Store 1,Sydney,NSW,Australia,2020-04-18
1,2,Sydney Store 2,Sydney,NSW,Australia,2020-08-15
2,3,Melbourne Store 3,Melbourne,VIC,Australia,2020-12-23
3,4,Melbourne Store 4,Melbourne,VIC,Australia,2022-05-04
4,5,Brisbane Store 5,Brisbane,QLD,Australia,2020-05-23
5,6,Perth Store 6,Perth,WA,Australia,2022-01-26
6,7,Adelaide Store 7,Adelaide,SA,Australia,2020-05-23
7,8,Canberra Store 8,Canberra,ACT,Australia,2020-10-25


## Sample output & distribution check

In [3]:
print(stores["city"].value_counts())

city
Sydney       2
Melbourne    2
Brisbane     1
Perth        1
Adelaide     1
Canberra     1
Name: count, dtype: int64


## Validation

In [4]:
assert stores["store_id"].is_unique
assert stores.isnull().sum().sum() == 0
assert (stores["city"] == cfg.ANOMALY_CITY).sum() >= 1, "Sydney must have at least one store"
print("All validation checks passed.")

All validation checks passed.


In [5]:
out_path = cfg.DATA_DIR / "stores.csv"
stores.to_csv(out_path, index=False)
print(f"Wrote {len(stores):,} rows to {out_path}")

Wrote 8 rows to /home/claude/project/eaida/data/raw/stores.csv


## Summary

Generated 8 stores across Sydney, Melbourne, Brisbane, Perth, Adelaide and
Canberra, each with an open date well before the order-history window. Saved
to `data/raw/stores.csv`.